#### Preparation

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
from torch.utils.data import DataLoader
from PIL import Image
import os
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm

In [2]:
DATA_DIR = r"C:\Users\yugil\Downloads\Cassava_Leaf_Datasets\Cassava_Leaf_Datasets\Classification"

WEIGHTS_DIR = r'../weights'

BATCH_SIZE = 32
FREEZE_EPOCHS = 8
FINETUNE_EPOCHS = 20

LR = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [3]:
weights = EfficientNet_V2_S_Weights.DEFAULT

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean,
        std=std
    )
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean,
        std=std
    )
])


In [4]:
train_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "val"),
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)


Classes: ['Healthy', 'Spider Mites', 'leaf blight', 'leaf spot']


In [5]:
model = efficientnet_v2_s(weights=weights)


Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to C:\Users\yugil/.cache\torch\hub\checkpoints\efficientnet_v2_s-dd5fe13b.pth
100%|██████████| 82.7M/82.7M [00:02<00:00, 42.8MB/s]


In [6]:
for param in model.parameters():
    param.requires_grad = False


In [7]:
in_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(in_features, num_classes)

model = model.to(DEVICE)

In [8]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.classifier.parameters(),
    lr=LR
)


In [9]:
def train_one_epoch(loader, epoch, epochs):
    model.train()
    
    train_loss = 0
    train_preds = []
    train_labels = []

    for images, labels in tqdm(loader, desc=f"Training Epoch {epoch+1}/{epochs}"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds = outputs.argmax(dim=1)

        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

        train_acc = accuracy_score(train_labels, train_preds)


    return train_loss / len(loader), train_acc


In [10]:
def validate(loader):
    model.eval()
    
    val_loss = 0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating"):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            preds = outputs.argmax(dim=1)

            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
        
        val_acc = accuracy_score(val_labels, val_preds)

        precision, recall, f1, _ = precision_recall_fscore_support(
        val_labels, val_preds, average='weighted'
        )   

    return precision, recall, f1, val_loss / len(loader), val_acc

#### Training 1 (Freeze Backbone)

In [11]:

print("\n-----------Starting Phase 1 Training-----------\n")

for epoch in range(FREEZE_EPOCHS):
    train_loss, train_acc = train_one_epoch(train_loader, epoch, FREEZE_EPOCHS)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{FREEZE_EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}\n\n")




-----------Starting Phase 1 Training-----------



Validating: 100%|██████████| 8/8 [00:30<00:00,  3.84s/it]



Epoch [1/8]
Train Loss: 1.3475 | Train Acc: 0.3688
Val Loss: 1.3119 | Val Acc: 0.4042
Precision: 0.4239 | Recall: 0.4042 | F1: 0.3912




Validating: 100%|██████████| 8/8 [00:29<00:00,  3.70s/it]



Epoch [2/8]
Train Loss: 1.2464 | Train Acc: 0.5583
Val Loss: 1.2325 | Val Acc: 0.5917
Precision: 0.6016 | Recall: 0.5917 | F1: 0.5897




...


Epoch [6/8]
Train Loss: 0.9767 | Train Acc: 0.7750
Val Loss: 1.0163 | Val Acc: 0.7500
Precision: 0.7533 | Recall: 0.7500 | F1: 0.7445




Validating: 100%|██████████| 8/8 [00:30<00:00,  3.76s/it]



Epoch [7/8]
Train Loss: 0.9162 | Train Acc: 0.8052
Val Loss: 0.9700 | Val Acc: 0.7792
Precision: 0.7840 | Recall: 0.7792 | F1: 0.7760




Validating: 100%|██████████| 8/8 [00:30<00:00,  3.78s/it]


Epoch [8/8]
Train Loss: 0.8705 | Train Acc: 0.8177
Val Loss: 0.9417 | Val Acc: 0.7667
Precision: 0.7655 | Recall: 0.7667 | F1: 0.7634




#### Training 2 (Fine-Tuning)

In [12]:
for param in model.features[-1].parameters():
    param.requires_grad = True


In [13]:
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5
)


In [14]:
best_val_acc = 0.0

print("\n-----------Starting Fine-tuning Stage-----------\n")

for epoch in range(FINETUNE_EPOCHS):
    train_loss, train_acc = train_one_epoch(train_loader, epoch, FINETUNE_EPOCHS)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{FINETUNE_EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}\n\n")


torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": class_names
}, os.path.join(WEIGHTS_DIR, "EfficientNet-v2.pth"))


-----------Starting Fine-tuning Stage-----------



Validating: 100%|██████████| 8/8 [00:29<00:00,  3.72s/it]



Epoch [1/20]
Train Loss: 0.8517 | Train Acc: 0.8260
Val Loss: 0.9182 | Val Acc: 0.7792
Precision: 0.7787 | Recall: 0.7792 | F1: 0.7763




Validating: 100%|██████████| 8/8 [00:29<00:00,  3.75s/it]



Epoch [2/20]
Train Loss: 0.8437 | Train Acc: 0.8000
Val Loss: 0.9167 | Val Acc: 0.7667
Precision: 0.7667 | Recall: 0.7667 | F1: 0.7612




...


Epoch [18/20]
Train Loss: 0.6336 | Train Acc: 0.8677
Val Loss: 0.7319 | Val Acc: 0.8208
Precision: 0.8205 | Recall: 0.8208 | F1: 0.8182




Validating: 100%|██████████| 8/8 [00:29<00:00,  3.72s/it]



Epoch [19/20]
Train Loss: 0.6280 | Train Acc: 0.8500
Val Loss: 0.7258 | Val Acc: 0.7958
Precision: 0.7990 | Recall: 0.7958 | F1: 0.7920




Validating: 100%|██████████| 8/8 [00:30<00:00,  3.77s/it]


Epoch [20/20]
Train Loss: 0.6202 | Train Acc: 0.8562
Val Loss: 0.7194 | Val Acc: 0.8333
Precision: 0.8363 | Recall: 0.8333 | F1: 0.8315




#### Prediction Sample

In [15]:
checkpoint = torch.load(r'../weights/EfficientNet-v2.pth')

model.load_state_dict(checkpoint["model_state_dict"])
class_names = checkpoint["class_names"]

model.eval()

def predict(image_path):
    image = Image.open(image_path).convert("RGB")
    image = val_transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return class_names[pred.item()]

# Example usage:
prediction = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Healthy\Healthy_val_17.jpg")
prediction1 = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Leaf Blight\leaf blight_val_17.jpg")
prediction2 = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Leaf Spot\leaf spot_val_17.jpg")
prediction3 = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Spider Mites\Spider Mites_val_17.jpg")
print(f"Healthy Predicted class: {prediction}")
print(f"Leaf Blight Predicted class: {prediction1}")
print(f"Leaf Spot Predicted class: {prediction2}")
print(f"Spider Mites Predicted class: {prediction3}")

C:\Users\yugil\AppData\Local\Temp\ipykernel_6308\4094652150.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(r'../weights/EfficientNet-v2.pth')


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\johnp\\Desktop\\Dataset\\Cassava_Leaf_Datasets\\Classification\\val\\Healthy\\Healthy_val_17.jpg'